# 3 · Hata ve zaman: kalibrasyon, kayma, backtest, serbest metin

Teknik raporun 8–10. bölümleri ([technical.tr.md](../reports/technical.tr.md)).

**Ne koşar, ne kayıttan gelir:**
- **Hafif betikler bu defterde koşar.** `script = run("<betik>")` betiğin [1]–[5] hücrelerini çalıştırır: veriyi ve
  kayıtlı model çıktılarını (OOF tahminleri, SHAP dosyası) okur, hesabı yapar; kaydet hücresi [6] çalışmaz.
  Ardından `check` sonucun `metrics/` kaydıyla aynı olup olmadığını gösterir. `script.res`, yüklenen tablolar
  (`script.listings` …) ve betiğin bütün fonksiyonları sonraki hücrelerde kullanılabilir.
- **Ağır betikler kayıttan okunur.** Model eğiten ya da çapraz doğrulama koşan betikler (`HEAVY`) için
  `recorded("<betik>", docs)` `metrics/` kaydını getirir.
- **Soru, veri, yöntem ve sızıntı notları** kartlardan (`analysis/cards/`), **figürler** derleyicilerin çizdiği
  `reports/figures/`'tan gelir.

Analiz kodu betiklerin kendisidir; ortak kod `analysis/lib` ve `builders/report_lib`'den gelir, burada kopyası yok. Bir
sonucu kalıcı olarak değiştirmek için betik değişir ve zincir koşulur; defter yeni kaydı okur. Defter dosya yazmaz.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis" / "lib").is_dir())
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "builders")]

import pandas as pd
from IPython.display import Markdown

from lib.records import card, check, flat, recorded, run, runs, show
from report_lib import metrics_view as MV

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 120)
SCRIPTS = ["08_residuals", "08_large_errors", "08_conformal_coverage", "09_drift", "09_backtest", "10_free_text"]
docs = MV.load_metrics(ROOT / "metrics")
MV.check_consistency(docs)
runs(docs, SCRIPTS)

,yazıldı,veri sonu,DB,run_id
betik,,,,
08_residuals,2026-11-22T18:55:40+03:00,2026-06-27,444123b1f1a9,601741517e09
08_large_errors,2026-11-22T18:55:38+03:00,2026-06-27,444123b1f1a9,601741517e09
08_conformal_coverage,2026-11-22T18:55:35+03:00,2026-06-27,444123b1f1a9,601741517e09
09_drift,2026-11-22T18:55:43+03:00,2026-06-27,444123b1f1a9,NaN
09_backtest,2026-09-28T22:40:54+03:00,2026-06-27,444123b1f1a9,601741517e09
10_free_text,2026-09-27T21:58:31+03:00,2026-06-27,444123b1f1a9,NaN


## 8. Kalibrasyon, artıklar ve zayıflık

### 08_residuals
[betik](../analysis/08_residuals.py) · [kart](../analysis/cards/08_residuals.json)

In [2]:
Markdown(card("08_residuals"))

**Soru:** OOF hataları nasıl dağılıyor; fiyat düzeyine göre yanlılık var mı?

**Veri:** load_clean() + 07_model_comparison'ın OOF tahminleri.

**Yöntem:** Hata bantları; tahmin edilen fiyatın çeyreklerine göre medyan hata, lira hatası payı ve sapma (gerçek fiyata göre gruplamak ortalamaya dönüş üretir); kalibrasyon eğimi; tahmin-gerçek ve artık figürlerinin noktaları.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `none` — Tekilleştirme (ad_id başına son kayıt, ORDER BY ad_id) fold'lardan önce; aynı ad_id iki fold'a düşemez.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Kendisi model kurmaz; ama okuduğu OOF tahminleri 07'nin erken durdurmasını taşır (fold'un kendi doğrulama kısmında): hatalar hafifçe iyimser olabilir. Etkisi ölçülmedi.
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Medyan artık dönemlere göre kayıyor (08_residuals); ileriye dönük hata 09_backtest'te ayrıca ölçülüyor.

In [3]:
script = run("08_residuals")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["08_residuals"])

median % error by predicted quartile | tahmin çeyreğine göre medyan % hata: [('Q1', 6.95), ('Q2', 4.97), ('Q3', 4.2), ('Q4', 3.55)]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [4]:
d, ed = m["domain"], m["error_drivers"]
show("tahmin ve gerçek (noktalar figürlerde)", flat({"pred_vs_true": d["pred_vs_true"], "residual_scatter": d["residual_scatter"]}))
show("tahmin fiyatı çeyreğine göre medyan |hata|", pd.DataFrame(d["quantile_error"], columns=["çeyrek", "medyan |hata| %"]))
show("tahmin fiyatı çeyreğine göre lira hatası",
     pd.DataFrame(ed["lira_quartile"], columns=["çeyrek", "ilan", "lira hatası payı %", "ortalama |hata| ₺", "medyan |hata| %"]))
show("tahmin fiyatı çeyreğine göre sapma (tahmin − gerçek)",
     pd.DataFrame(ed["pred_quartile"], columns=["çeyrek", "ilan", "ortalama sapma ₺", "medyan sapma ₺"]))
show("kalibrasyon (gerçek ~ a + b·tahmin)", flat(ed["calibration"]))
show("hata dağılımı: özet", flat(m["report"]))
show("hata bantları", pd.DataFrame(m["report"]["err_bands"], columns=["alt %", "üst %", "ilan", "pay %"]))
show("model başına (en az 5 ilan)", pd.DataFrame(d["residual_vs_n"], columns=["ilan", "medyan |hata| %", "medyan fiyat"]))

**tahmin ve gerçek (noktalar figürlerde)**

,değer
pred_vs_true.ideal_line,"345000.0, 6500000.0"
pred_vs_true.r2,0.9745
pred_vs_true.n,29988
pred_vs_true.note,"OOF tahmin (sızıntısız), TÜM noktalar. Frontend: yoğunluk/hexbin veya düşük-opacity ile render (30K nokta ham scatte..."
residual_scatter.mean_resid_pct,-0.47
residual_scatter.std_resid_pct,9.27
residual_scatter.n,29988
residual_scatter.note,"Artık% = (gerçek-tahmin)/gerçek, TÜM noktalar. Ortalama sıfıra yakın, ama medyan artık dönemlere göre %-3.27 ile %+1..."


**tahmin fiyatı çeyreğine göre medyan |hata|**

,çeyrek,medyan |hata| %
0,Q1,6.95
1,Q2,4.97
2,Q3,4.20
3,Q4,3.55


**tahmin fiyatı çeyreğine göre lira hatası**

,çeyrek,ilan,lira hatası payı %,ortalama |hata| ₺,medyan |hata| %
0,Q1,7497,17.6,77305.0,6.95
1,Q2,7497,19.9,87321.0,4.97
2,Q3,7497,23.6,103608.0,4.20
3,Q4,7497,38.9,170870.0,3.55


... (6 outputs left out)

**hata bantları**

,alt %,üst %,ilan,pay %
0,0,5.0,15775,52.604375
1,5,10.0,8221,27.414299
2,10,20.0,4784,15.953048
3,20,NaN,1208,4.028278


**model başına (en az 5 ilan)**

,ilan,medyan |hata| %,medyan fiyat
0,25,8.99,870000.0
1,15,4.17,835000.0
2,27,7.03,880000.0
3,30,5.54,967000.0
4,6,9.33,970000.0
...,...,...,...
433,6,11.31,1179500.0
434,15,3.69,2490000.0
435,10,6.42,2010000.0
436,22,7.48,4697500.0


![tr-08-pred-vs-true](../reports/figures/tr-08-pred-vs-true.png)

![tr-09-residual](../reports/figures/tr-09-residual.png)

![tr-28-residual-lira](../reports/figures/tr-28-residual-lira.png)

![tr-26-error-hist](../reports/figures/tr-26-error-hist.png)

![tr-10-quartile-error](../reports/figures/tr-10-quartile-error.png)

![tr-27-quartile-lira](../reports/figures/tr-27-quartile-lira.png)

### 08_large_errors
[betik](../analysis/08_large_errors.py) · [kart](../analysis/cards/08_large_errors.json)

In [5]:
Markdown(card("08_large_errors"))

**Soru:** Büyük hatalar (|OOF artık| > %20) nereden geliyor: emsal sayısı, model, yaş, segment, tarama, lira ölçeğinde en büyük hataların yeri, katalog uyuşmazlığı?

**Veri:** load_clean() (29.988 ilan) ve 07_model_comparison'ın OOF tahminleri (data/analysis/oof.parquet).

**Yöntem:** Büyük hata oranı emsal sayısına, modele, yaşa (her yaş ve her kesim), F/S segmente, son görüldüğü taramaya ve canlı/kaybolan ayrımına göre; en büyük 100 lira hatasının düşük/fazla, en pahalı çeyrek ve performans ailesi dağılımı; kendi modelinin medyanından sapan motor değerleri; notlarının dayandığı karşılaştırmalarla üç örnek (her koşuda hesaplanır).

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — Kırılımlar ilan özniteliklerinden; hata OOF tahminden.
- bütün veriye uydurma: `n/a`
- zamansal: `present` — Model zamanı görmüyor; taramaya göre kayma burada ölçülüyor, düzeltilmiyor (09 dönem etkisini ayrı ölçer).

In [6]:
script = run("08_large_errors")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["08_large_errors"])

large errors | büyük hata 1208 (768 over / 440 under)


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [7]:
ed = m["error_drivers"]
show("büyük hata: özet", flat(ed))
show("emsal sayısına göre", pd.DataFrame(ed["by_model_year_n"]))
show("model başına ilan kovasına göre", pd.DataFrame(ed["per_model_buckets"]))
show("son görüldüğü taramaya göre", pd.DataFrame(ed["by_snapshot"]))
show("yaşa göre büyük hata oranı", pd.DataFrame(ed["age_sensitivity"], columns=["yaş", "ilan", "büyük hata %"]))
show("yaş kesimleri", pd.DataFrame(ed["age_cuts"], columns=["kesim t", "≥t ilan", "≥t büyük hata %", "<t büyük hata %", "oran"]))
show("örnek ilanlar", pd.DataFrame(ed["examples"]))
show("model başına (bütün modeller)", pd.DataFrame(ed["per_model_error"], columns=["ilan", "medyan |hata| %"]))

**büyük hata: özet**

,değer
threshold_pct,20.0
overall.n,29988
overall.n_big,1208
overall.n_over,768
overall.n_under,440
...,...
spec_outliers.blind_spot.model,307
spec_outliers.n,15
spec_outliers.pct,0.05
spec_outliers.median_error_pct,12.8


**emsal sayısına göre**

,bin,n,big_pct,over_pct,under_pct
0,1,610,19.02,10.66,8.36
1,2–4,1584,10.48,6.19,4.29
2,5–19,5732,4.73,3.02,1.71
3,20–99,16505,2.99,1.94,1.05
4,100+,5557,2.90,2.02,0.88


**model başına ilan kovasına göre**

,bin,lo,hi,n_models,n_listings,median_of_medians
0,1,1,1,126,126,8.89
1,2–4,2,4,181,506,8.47
2,5–19,5,19,188,1876,5.99
3,20–99,20,99,171,7946,4.71
4,100+,100,1000000000,79,19534,4.58


... (6 outputs left out)

**örnek ilanlar**

,key,name,year,km,price,pred,resid_pct,n_model,n_model_year,compare
0,"[640i, 2011, 5600000]",BMW 640i,2011,160000,5600000.0,3236953.0,42.2,3,2,"{'series_n': 51, 'others': [[2011, 3550000.0], [2012, 2485000.0]], 'peer': None}"
1,"[4.2 FSI Quattro R-tronic, 2008, 4690000]",Audi 4.2 FSI Quattro R-tronic (R8),2008,112550,4690000.0,2744815.0,41.5,1,1,"{'series_n': 1, 'others': [], 'peer': {'model': 'S5 4.2 FSI Quattro', 'n': 4, 'median': 2622500.0}}"
2,"[750i Long, 2007, 1190000]",BMW 750i Long,2007,271000,1190000.0,2831842.0,-138.0,2,1,"{'series_n': 244, 'others': [[2009, 5300000.0]], 'peer': {'model_prefix': '730d', 'same_year': True, 'n': 15, 'media..."


**model başına (bütün modeller)**

,ilan,medyan |hata| %
0,25,8.99
1,15,4.17
2,27,7.03
3,30,5.54
4,6,9.33
...,...,...
740,22,7.48
741,1,7.70
742,3,44.79
743,1,22.19


![tr-11-n-vs-error](../reports/figures/tr-11-n-vs-error.png)

### 08_conformal_coverage
[betik](../analysis/08_conformal_coverage.py) · [kart](../analysis/cards/08_conformal_coverage.json)

In [8]:
Markdown(card("08_conformal_coverage"))

**Soru:** %90 conformal aralık tahmin edilen fiyatın her bandında ilanların %90'ını kapsıyor mu; banda göre ayrı kalibre edilince (Mondrian) ne değişiyor?

**Veri:** load_clean() + 07_model_comparison'ın OOF tahminleri.

**Yöntem:** Aralık expm1(tahmin_log ± q). Çapraz kalibrasyon: her katın q'su yalnız öteki katların OOF log hatalarından (sonlu örneklem yüzdeliği). İki kol: tek q ve tahmin edilen fiyatın çeyrek bandı başına q (ön kayıt plans/08-mondrian-coverage). Bant başına kapsama ve medyan aralık genişliği.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `none` — Tekilleştirme (ad_id başına son kayıt, ORDER BY ad_id) fold'lardan önce; aynı ad_id iki fold'a düşemez.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `none` — Çapraz kalibreli: hiçbir ilanın kendi hatası kendi aralığını kalibre etmez (test_calibration_never_sees_the_scored_fold). Ama katlar rastgele olduğu için genel ve bant başına kapsama yine neredeyse tanım gereği ~%90; ileri kullanımdaki kapsama 09_backtest'te (plans/09-forward-coverage). OOF hataları 07'nin erken durdurması yüzünden hafifçe iyimser olabilir.
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Medyan artık dönemlere göre kayıyor (08_residuals); ileriye dönük hata 09_backtest'te ayrıca ölçülüyor.

In [9]:
script = run("08_conformal_coverage")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["08_conformal_coverage"])

overall | genel: 89.99% · [('Q1', 80.99, 89.96), ('Q2', 90.98, 90.04), ('Q3', 93.25, 90.01), ('Q4', 94.72, 90.04)]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [10]:
mo = m["report"]["mondrian"]
show("aralık: özet", flat({"conformal": m["domain"]["conformal"], "report": m["report"]}))
show("tahmin fiyatı bandına göre kapsama ve genişlik", pd.DataFrame(mo["by_band"], columns=mo["columns"]))

**aralık: özet**

,değer
conformal.coverage_target,90
conformal.note,Bantlar tahmin edilen fiyatın çeyrekleri; kapsama çapraz kalibreli (her katın q'su öteki katlardan). by_quantile = t...
report.conformal_q,0.14123
report.conformal_all,89.99
report.q_bounds,"1154371.0315541304, 1532721.5318234982, 2256098.228932405"
report.mondrian.accept_band,"88, 92"
report.mondrian.columns,"band, n, coverage_global, coverage_band, width_pct_global, width_pct_band"


**tahmin fiyatı bandına göre kapsama ve genişlik**

,band,n,coverage_global,coverage_band,width_pct_global,width_pct_band
0,Q1,7497,80.99,89.96,28.3,37.1
1,Q2,7497,90.98,90.04,28.3,27.2
2,Q3,7497,93.25,90.01,28.3,24.4
3,Q4,7497,94.72,90.04,28.3,21.4


![tr-12-coverage](../reports/figures/tr-12-coverage.png)

## 9. Zaman — dönem etkisi, dağılım kayması ve backtest

### 09_drift
[betik](../analysis/09_drift.py) · [kart](../analysis/cards/09_drift.json)

In [11]:
Markdown(card("09_drift"))

**Soru:** Piyasa taramalar arasında kaydı mı?

**Veri:** load_clean(all_snapshots=True) (her taramanın o gün yayındaki bütün ilanları) + load_clean().

**Yöntem:** Fiyat dağılımları farkın büyüklüğüyle (KS istatistiği, PSI, EMD) ve ortak ilan payıyla karşılaştırılır; taramalar ilan paylaştığı için p-değeri yayımlanmaz. (model, yıl) hücre medyanlarının değişimi.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Aynı ilan birden çok taramada: bir anlamlılık testi bağımsız örnek varsayardı. Bu yüzden yalnız farkın büyüklüğü ve ortak ilan payı yayımlanıyor, p-değeri yok.
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [12]:
script = run("09_drift")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["09_drift"])

max PSI: 0.0049 · cell shift | hücre kayması: [['01-27', 0.0, 776], ['03-21', 0.85, 728], ['06-27', 1.98, 701]]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [13]:
show("tarama çiftleri", pd.DataFrame(m["domain"]["drift"]["all_pairs"], columns=["çift", "KS", "PSI", "EMD (₺)", "ortak ilan %"]))
show("(model, yıl) hücre medyanlarının değişimi (ilk taramaya göre)",
     pd.DataFrame(m["error_drivers"]["period_shift"]["live"], columns=["tarama", "medyan değişim %", "hücre"]))
show("not ve ayarlar", flat({"note": m["domain"]["drift"]["note"], "min_cell_n": m["error_drivers"]["period_shift"]["min_cell_n"]}))

**tarama çiftleri**

,çift,KS,PSI,EMD (₺),ortak ilan %
0,01-18→01-27,0.0055,0.0004,10109.0,76.1
1,01-18→03-21,0.0173,0.0015,20560.0,30.2
2,01-18→06-27,0.0309,0.0049,48059.0,9.1
3,01-27→03-21,0.0161,0.0011,15717.0,36.1
4,01-27→06-27,0.0301,0.0038,39115.0,10.8
5,03-21→06-27,0.0157,0.0017,28620.0,21.1


**(model, yıl) hücre medyanlarının değişimi (ilk taramaya göre)**

,tarama,medyan değişim %,hücre
0,01-27,0.00,776
1,03-21,0.85,728
2,06-27,1.98,701


**not ve ayarlar**

,değer
note,"KS=maks dağılım farkı, PSI<0.10 güvenli/>0.25 retrain, EMD=kayma mesafesi (₺). hist.edges = bin kenarları (39 bin, 4..."
min_cell_n,3


![tr-13-drift-hist](../reports/figures/tr-13-drift-hist.png)

![tr-14-drift-kde](../reports/figures/tr-14-drift-kde.png)

### 09_backtest
[betik](../analysis/09_backtest.py) · [kart](../analysis/cards/09_backtest.json)

In [14]:
Markdown(card("09_backtest"))

**Soru:** Geçmişte eğitilen model sonraki taramayı ne kadar iyi fiyatlıyor?

**Veri:** load_clean(all_snapshots=True): her taramanın her satırı; testte yalnız eğitimde görülmemiş ad_id'ler.

**Yöntem:** Dört kol, hepsi manşet kurulumla (cv.fold_matrices + cv.LGB_PARAMS): single (bir taramada eğit, sonrakilerin yeni ilanlarında test), cumulative (t'ye kadar hepsinde eğit), insample ve per_snapshot (5-fold OOF, karşılaştırma için; insample'ın son bloğu manşet OOF'un kendisi). İleri kollarda ağaç = eğitim kümesinde erken durdurmalı iç 5-fold CV'nin medyanı. Her ileri satıra test ilanlarını modele göre yeniden örnekleyen bootstrap'ten %95 aralık; single ile cumulative aynı ilanlarda eşli karşılaştırılır. Ufuk ayrıca sabit bir test kümesinde (son taramada yeni ilanlar) önceki her tek taramayla eğitilen modellerle eşli ölçülür. İleri kapsama (ön kayıt plans/09-forward-coverage): q eğitim kümesinin kendi OOF hatalarından (servis tarifi), bant sınırları eğitim OOF tahminlerinden; tek q ve bant başına q, test ilanlarında bant başına ve emsalsiz ilanlarda kapsama.

**Bölme:** `temporal`

**Sızıntı:**
- aynı ilan iki tarafta: `none` — İleriye dönük kollarda test yalnız eğitimde hiç görülmemiş ad_id'ler.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `none` — Her kol yalnız kendi eğitim bloğunda kurulur; ileri kollarda ağaç sayısını eğitim kümesinin iç CV'si seçer, test taraması erken durdurmada kullanılmaz.
- zamansal: `none` — single/cumulative zaman sırasını korur; insample/per_snapshot bilerek düz KFold ve protokolde öyle etiketli.

In [15]:
m = recorded("09_backtest", docs)

**Kayıttan okunuyor** (ağır betik: her tarama ve eğitim kümesi için model eğitimi).

In [16]:
bt = m["methodology"]["backtest"]
fw, pa = bt["columns"]["forward"], bt["columns"]["paired"]
show("tek tarama ile eğitim → sonraki taramalar", pd.DataFrame(bt["single"], columns=fw))
show("biriken taramalarla eğitim → sonraki taramalar", pd.DataFrame(bt["cumulative"], columns=fw))
show("tek ve biriken, aynı ilanlarda (eşli)", pd.DataFrame(bt["paired"], columns=pa))
show("ufuk: sabit test kümesi", flat(bt["horizon"]))
show("ufuk: eğitim taraması başına MAPE", pd.DataFrame(bt["horizon"]["rows"], columns=["eğitim", "MAPE", "alt", "üst"]))
show("ufuk: en yeni taramaya göre fark", pd.DataFrame(bt["horizon"]["vs_latest"], columns=["eğitim", "fark", "alt", "üst"]))
show("ileriye kapsama", pd.DataFrame(bt["forward_coverage"]))
show("zaman boyutu olmadan: biriken blok içinde 5-fold", pd.DataFrame(bt["insample"], columns=["blok", "MAPE", "ilan"]))
show("zaman boyutu olmadan: tarama içinde 5-fold", pd.DataFrame(bt["per_snapshot"], columns=["tarama", "MAPE", "ilan"]))
show("protokol", flat(bt["protocol"]))

**tek tarama ile eğitim → sonraki taramalar**

,train,test,MAPE,n,ci_lo,ci_hi,trees
0,01-18,01-27,6.29,2960,5.90,6.73,163
1,01-18,03-21,6.59,8182,6.24,6.96,163
2,01-18,06-27,7.34,10529,6.95,7.73,163
3,01-27,03-21,6.54,7413,6.18,6.91,161
4,01-27,06-27,7.18,10313,6.82,7.60,161
5,03-21,06-27,6.90,9099,6.49,7.33,195


**biriken taramalarla eğitim → sonraki taramalar**

,train,test,MAPE,n,ci_lo,ci_hi,trees
0,→01-18,01-27,6.29,2960,5.90,6.73,163
1,→01-18,03-21,6.59,8182,6.24,6.96,163
2,→01-18,06-27,7.34,10529,6.95,7.73,163
3,→01-27,03-21,6.47,7238,6.09,6.83,162
4,→01-27,06-27,7.23,10257,6.85,7.64,162
5,→03-21,06-27,6.78,8889,6.42,7.17,211


**tek ve biriken, aynı ilanlarda (eşli)**

,test,single_train,cumulative_train,n_common,single_MAPE,cumulative_MAPE,difference,ci_lo,ci_hi
0,03-21,01-27,→01-27,7238,6.52,6.47,-0.04,-0.12,0.03
1,06-27,01-27,→01-27,10257,7.17,7.23,0.06,-0.01,0.12
2,06-27,03-21,→03-21,8889,6.89,6.78,-0.11,-0.20,-0.02


... (10 outputs left out)

**zaman boyutu olmadan: tarama içinde 5-fold**

,tarama,MAPE,ilan
0,01-18,6.91,10901
1,01-27,6.92,11254
2,03-21,6.93,11478
3,06-27,7.09,11526


**protokol**

,değer
leak_free,"single, cumulative"
plain_kfold,"insample, per_snapshot"
same_experiment,cumulative first block (<= first snapshot) equals the single first block
setup,headline: cv.fold_matrices + cv.LGB_PARAMS; forward trees = median of an inner 5-fold early-stopping CV on the train...
interval.level,0.95
interval.n_boot,1000
interval.resample,model


![tr-15-backtest](../reports/figures/tr-15-backtest.png)

## 10. Serbest metin: ölçüldü, dahil edilmedi

### 10_free_text
[betik](../analysis/10_free_text.py) · [kart](../analysis/cards/10_free_text.json)

In [17]:
Markdown(card("10_free_text"))

**Soru:** Satıcı açıklaması modele eklenince OOF doğruluğu ne kadar değişir? (ön kayıt: plans/10-text-contribution)

**Veri:** load_clean() (modelin ilanları) ve açıklamaları (description_text); bu koşunun model OOF'u (07_model_comparison, cv.load_oof).

**Yöntem:** Taban kol = modelin OOF'u. Metin kolu = aynı 5 fold, öznitelikler ve LightGBM ayarları + açıklamanın kelime TF-IDF'i (1–2 gram, min_df 5) ve SVD 50, ikisi de eğitim fold'unda kurulur. İki kolun log R², MAPE ve MAE'si ve farkları; H1: ΔR² < 0.005 ve MAPE iyileşmesi < 0,2 puan.

**Bölme:** `kfold_oof`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Modelle aynı fold'lar: farklı ad_id ile yeniden ilan edilen aynı araç fold'lar arasına düşebilir (01_dedup_leakage %0,46–%4,26); iki kolu eşit etkiler.
- hedef öznitelikte: `none` — Öznitelikler modelinkiler artı açıklama metni; fiyattan türeyen alan yok. Açıklama fiyatı yazabilir; ölçülen fark bunu da içerir.
- bütün veriye uydurma: `present` — TF-IDF ve SVD yalnız eğitim fold'unda kurulur (test_text_features_fit_on_train_only). Erken durdurma modeldeki gibi fold'un doğrulama kısmında; iki kolu eşit etkiler.
- zamansal: `present` — KFold dönemleri karıştırır, modeldeki gibi; iki kol aynı bölmeyi kullanır.

In [18]:
m = recorded("10_free_text", docs)

**Kayıttan okunuyor** (ağır betik: açıklama metniyle 5-fold çapraz doğrulama).

In [19]:
ta = m["report"]["text_ablation"]
show("açıklama metni eklenince", flat(ta))

**açıklama metni eklenince**

,değer
r2_log_base,0.9699
r2_log_text,0.9707
delta_r2,0.0008
mape_base,6.49
mape_text,6.41
mae_base,109776
mae_text,108770
unexplained_share_pct,2.6
n,29988
folds,5
